In [4]:
#import triton_python_backend_utils as pb_utils
import numpy as np
import json


def gen_annotation():
    ions = [
        "y",
        "b",
    ]
    charges = [1, 2, 3]
    positions = [x for x in range(1, 30)]
    annotation = []
    for pos in positions:
        for ion in ions:
            for charge in charges:
                annotation.append(f"{ion}{pos}+{charge}")
    return np.array(annotation).astype(np.object_)


In [6]:
gen_annotation ()

array(['y1+1', 'y1+2', 'y1+3', 'b1+1', 'b1+2', 'b1+3', 'y2+1', 'y2+2',
       'y2+3', 'b2+1', 'b2+2', 'b2+3', 'y3+1', 'y3+2', 'y3+3', 'b3+1',
       'b3+2', 'b3+3', 'y4+1', 'y4+2', 'y4+3', 'b4+1', 'b4+2', 'b4+3',
       'y5+1', 'y5+2', 'y5+3', 'b5+1', 'b5+2', 'b5+3', 'y6+1', 'y6+2',
       'y6+3', 'b6+1', 'b6+2', 'b6+3', 'y7+1', 'y7+2', 'y7+3', 'b7+1',
       'b7+2', 'b7+3', 'y8+1', 'y8+2', 'y8+3', 'b8+1', 'b8+2', 'b8+3',
       'y9+1', 'y9+2', 'y9+3', 'b9+1', 'b9+2', 'b9+3', 'y10+1', 'y10+2',
       'y10+3', 'b10+1', 'b10+2', 'b10+3', 'y11+1', 'y11+2', 'y11+3',
       'b11+1', 'b11+2', 'b11+3', 'y12+1', 'y12+2', 'y12+3', 'b12+1',
       'b12+2', 'b12+3', 'y13+1', 'y13+2', 'y13+3', 'b13+1', 'b13+2',
       'b13+3', 'y14+1', 'y14+2', 'y14+3', 'b14+1', 'b14+2', 'b14+3',
       'y15+1', 'y15+2', 'y15+3', 'b15+1', 'b15+2', 'b15+3', 'y16+1',
       'y16+2', 'y16+3', 'b16+1', 'b16+2', 'b16+3', 'y17+1', 'y17+2',
       'y17+3', 'b17+1', 'b17+2', 'b17+3', 'y18+1', 'y18+2', 'y18+3',
       'b18

In [9]:
import re

def find_crosslinker_position(string):
    pattern = r"K\[UNIMOD:\d+\]"
    match = re.search(pattern, string)
    if match:
        crosslinker_position = match.start() + 1
        return crosslinker_position
    else:
        return None


In [15]:
string = "GGKLFKSDFSK[UNIMOD:23]"
position = find_crosslinker_position(string)
print(position)


11


In [22]:
peak_pos_xl_cms2("KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK",2)

NameError: name 'constants' is not defined

In [20]:
import spectrum_fundamentals.constants as constants
from spectrum_fundamentals.annotation.annotation import peak_pos_xl_cms2
#import spectrum_fundamentals.annotation.annotation as annotation
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:

#import triton_python_backend_utils as pb_utils
import numpy as np
import json
import re



def find_crosslinker_position(peptide_sequences_1: str):
    pattern = r"K\[UNIMOD:\d+\]"
    match = re.search(pattern, peptide_sequences_1)
    if match:
        crosslinker_position = match.start() + 1
        return crosslinker_position
    else:
        return None

def gen_annotation_linear_pep():
    ions = [
        "y",
        "b",    
    ]
    charges = ["1", "2", "3"]
    positions = [x for x in range(1, 30)]
    annotation = []
    for pos in positions:
        for ion in ions:
            for charge in charges:
                annotation.append(ion + str(pos) + "+" + charge)
    return annotation

def gen_annotation_xl(crosslinker_position: int):
    annotations = gen_annotation_linear_pep()
    annotation = np.concatenate((annotations, annotations))
    annotation = annotation.tolist()
    peaks_range, peaks_y, peaks_b, peaks_yshort, peaks_bshort, peaks_ylong, peaks_blong = peak_pos_xl_cms2("K" * 30, crosslinker_position)
    for pos in peaks_yshort:
         annotation[pos] = "y_short" + annotation[pos][1:]
    for pos in peaks_bshort:
         annotation[pos] = "b_short" + annotation[pos][1:]
    for pos in peaks_ylong:
         annotation[pos] = "y_long" + annotation[pos][1:]
    for pos in peaks_blong:
         annotation[pos] = "b_long" + annotation[pos][1:]
    pos_none = [num + 174 for num in peaks_y] + [num + 174 for num in peaks_b]
    for pos in pos_none:
         annotation[pos] = "None"
    return np.array(annotation).astype(np.object_)
 
class TritonPythonModel:
    def initialize(self, args):
        self.model_config = json.loads(args["model_config"])
        output0_config = pb_utils.get_output_config_by_name(
            self.model_config, "annotation"
        )
        self.output_dtype = pb_utils.triton_string_to_numpy(output0_config["data_type"])

    def execute(self, requests):
        responses = []
        for request in requests:
            batchsize = (
                pb_utils.get_input_tensor_by_name(request, "precursor_charges")
                .as_numpy()
                .shape[0]
            )
            annotation = np.tile(gen_annotation(), batchsize).reshape((-1, 174))
            t = pb_utils.Tensor("annotation", annotation)
            responses.append(pb_utils.InferenceResponse(output_tensors=[t]))
        return responses

    def finalize(self):
        pass

In [154]:
gen_annotation_xl(2)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


AttributeError: 'numpy.ndarray' object has no attribute 'peak_pos_xl_cms2'

In [38]:
print(gen_annotation_linear_pep())
annotation  = np.concatenate(gen_annotation_linear_pep(), gen_annotation_linear_pep())

['y1+1' 'y1+2' 'y1+3' 'b1+1' 'b1+2' 'b1+3' 'y2+1' 'y2+2' 'y2+3' 'b2+1'
 'b2+2' 'b2+3' 'y3+1' 'y3+2' 'y3+3' 'b3+1' 'b3+2' 'b3+3' 'y4+1' 'y4+2'
 'y4+3' 'b4+1' 'b4+2' 'b4+3' 'y5+1' 'y5+2' 'y5+3' 'b5+1' 'b5+2' 'b5+3'
 'y6+1' 'y6+2' 'y6+3' 'b6+1' 'b6+2' 'b6+3' 'y7+1' 'y7+2' 'y7+3' 'b7+1'
 'b7+2' 'b7+3' 'y8+1' 'y8+2' 'y8+3' 'b8+1' 'b8+2' 'b8+3' 'y9+1' 'y9+2'
 'y9+3' 'b9+1' 'b9+2' 'b9+3' 'y10+1' 'y10+2' 'y10+3' 'b10+1' 'b10+2'
 'b10+3' 'y11+1' 'y11+2' 'y11+3' 'b11+1' 'b11+2' 'b11+3' 'y12+1' 'y12+2'
 'y12+3' 'b12+1' 'b12+2' 'b12+3' 'y13+1' 'y13+2' 'y13+3' 'b13+1' 'b13+2'
 'b13+3' 'y14+1' 'y14+2' 'y14+3' 'b14+1' 'b14+2' 'b14+3' 'y15+1' 'y15+2'
 'y15+3' 'b15+1' 'b15+2' 'b15+3' 'y16+1' 'y16+2' 'y16+3' 'b16+1' 'b16+2'
 'b16+3' 'y17+1' 'y17+2' 'y17+3' 'b17+1' 'b17+2' 'b17+3' 'y18+1' 'y18+2'
 'y18+3' 'b18+1' 'b18+2' 'b18+3' 'y19+1' 'y19+2' 'y19+3' 'b19+1' 'b19+2'
 'b19+3' 'y20+1' 'y20+2' 'y20+3' 'b20+1' 'b20+2' 'b20+3' 'y21+1' 'y21+2'
 'y21+3' 'b21+1' 'b21+2' 'b21+3' 'y22+1' 'y22+2' 'y22+3' 'b22+1' 

TypeError: only integer scalar arrays can be converted to a scalar index

In [11]:
import numpy as np
def gen_annotation_linear_pep():
    ions = [
        "y",
        "b",    
    ]
    charges = ["1", "2", "3"]
    positions = [x for x in range(1, 30)]
    annotation = []
    for pos in positions:
        for ion in ions:
            for charge in charges:
                annotation.append(ion + str(pos) + "+" + charge)
    return annotation
    
def gen_annotation_xl(crosslinker_position: int):
    annotations = gen_annotation_linear_pep()
    annotation = np.concatenate((annotations, annotations))
    annotation = annotation.tolist()
    peaks_range, peaks_y, peaks_b, peaks_yshort, peaks_bshort, peaks_ylong, peaks_blong = annotation.peak_pos_xl_cms2("K" * 30, crosslinker_position)
    for pos in peaks_yshort:
         annotation[pos] = "y_short" + annotation[pos][1:]
    for pos in peaks_bshort:
         annotation[pos] = "b_short" + annotation[pos][1:]
    for pos in peaks_ylong:
         annotation[pos] = "y_long" + annotation[pos][1:]
    for pos in peaks_blong:
         annotation[pos] = "b_long" + annotation[pos][1:]
    pos_none = [num + 174 for num in peaks_y] + [num + 174 for num in peaks_b]
    for pos in pos_none:
         annotation[pos] = "None"
    return np.array(annotation).astype(np.object_)


In [22]:
#print(annotation[0])
#print(annotation[0][1])
gen_annotation_xl(30)


array(['y_short1+1', 'y_short1+2', 'y_short1+3', 'b1+1', 'b1+2', 'b1+3',
       'y_short2+1', 'y_short2+2', 'y_short2+3', 'b2+1', 'b2+2', 'b2+3',
       'y_short3+1', 'y_short3+2', 'y_short3+3', 'b3+1', 'b3+2', 'b3+3',
       'y_short4+1', 'y_short4+2', 'y_short4+3', 'b4+1', 'b4+2', 'b4+3',
       'y_short5+1', 'y_short5+2', 'y_short5+3', 'b5+1', 'b5+2', 'b5+3',
       'y_short6+1', 'y_short6+2', 'y_short6+3', 'b6+1', 'b6+2', 'b6+3',
       'y_short7+1', 'y_short7+2', 'y_short7+3', 'b7+1', 'b7+2', 'b7+3',
       'y_short8+1', 'y_short8+2', 'y_short8+3', 'b8+1', 'b8+2', 'b8+3',
       'y_short9+1', 'y_short9+2', 'y_short9+3', 'b9+1', 'b9+2', 'b9+3',
       'y_short10+1', 'y_short10+2', 'y_short10+3', 'b10+1', 'b10+2',
       'b10+3', 'y_short11+1', 'y_short11+2', 'y_short11+3', 'b11+1',
       'b11+2', 'b11+3', 'y_short12+1', 'y_short12+2', 'y_short12+3',
       'b12+1', 'b12+2', 'b12+3', 'y_short13+1', 'y_short13+2',
       'y_short13+3', 'b13+1', 'b13+2', 'b13+3', 'y_short14+1',
     

In [158]:
def gen_annotation_linear_pep():
    ions = [
        "y",
        "b",    
    ]
    charges = ["1", "2", "3"]
    positions = [x for x in range(1, 30)]
    #print(positions)
    #positions = [str(num) for num in positions]
    print(positions)
    annotation = []
    for pos in positions:
        for ion in ions:
            for charge in charges:
                annotation.append(ion + str(pos) + "+" + charge)
    return annotation

In [159]:
gen_annotation_linear_pep()


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


['y1+1',
 'y1+2',
 'y1+3',
 'b1+1',
 'b1+2',
 'b1+3',
 'y2+1',
 'y2+2',
 'y2+3',
 'b2+1',
 'b2+2',
 'b2+3',
 'y3+1',
 'y3+2',
 'y3+3',
 'b3+1',
 'b3+2',
 'b3+3',
 'y4+1',
 'y4+2',
 'y4+3',
 'b4+1',
 'b4+2',
 'b4+3',
 'y5+1',
 'y5+2',
 'y5+3',
 'b5+1',
 'b5+2',
 'b5+3',
 'y6+1',
 'y6+2',
 'y6+3',
 'b6+1',
 'b6+2',
 'b6+3',
 'y7+1',
 'y7+2',
 'y7+3',
 'b7+1',
 'b7+2',
 'b7+3',
 'y8+1',
 'y8+2',
 'y8+3',
 'b8+1',
 'b8+2',
 'b8+3',
 'y9+1',
 'y9+2',
 'y9+3',
 'b9+1',
 'b9+2',
 'b9+3',
 'y10+1',
 'y10+2',
 'y10+3',
 'b10+1',
 'b10+2',
 'b10+3',
 'y11+1',
 'y11+2',
 'y11+3',
 'b11+1',
 'b11+2',
 'b11+3',
 'y12+1',
 'y12+2',
 'y12+3',
 'b12+1',
 'b12+2',
 'b12+3',
 'y13+1',
 'y13+2',
 'y13+3',
 'b13+1',
 'b13+2',
 'b13+3',
 'y14+1',
 'y14+2',
 'y14+3',
 'b14+1',
 'b14+2',
 'b14+3',
 'y15+1',
 'y15+2',
 'y15+3',
 'b15+1',
 'b15+2',
 'b15+3',
 'y16+1',
 'y16+2',
 'y16+3',
 'b16+1',
 'b16+2',
 'b16+3',
 'y17+1',
 'y17+2',
 'y17+3',
 'b17+1',
 'b17+2',
 'b17+3',
 'y18+1',
 'y18+2',
 'y18+3',
 'b1